# Part C: Cut-offs and flags

**What this notebook does, in order**
1. Loads the saved predictions (from the predictors notebook) and the labels (from Part A). Keeps only the 200 labeled buildings.
2. Measures each predictor's error on every hour.
3. Learns each building's normal error from the validation period only, with labeled anomaly hours left out.
4. Scores every hour: how unusual is this error for this building?
5. Picks one cut-off per predictor so that all predictors raise the same false-alarm rate on normal validation hours.
6. Builds the comparison method: a fixed percentile of each building's own training use.
7. **Freezes** everything to files. Nothing is changed after this point.
8. Reloads the frozen files and produces flags for validation and test.
9. Checks that the flag rates on normal validation hours are close to the target.

**Why the cut-offs are frozen first:** if cut-offs were adjusted after looking at test results, the test results would no longer be an honest check. The notebook reloads the saved files in step 8 on purpose, so the flags can only come from the frozen values.

**Do not report flag rates as findings.** They are set by the cut-off, not by how good a predictor is. The real comparison happens in the evaluation notebook (Part D).

Run the cells in order and read each printed result before moving on. Cells marked **CHANGE** may need your file or column names.

## 0. Settings (CHANGE)

Paths are relative to the folder this notebook is in. The predictions file must have one row per building and hour, with the real reading and one prediction column per predictor. Save predictions in the **same unit as the reading (kWh)**, even if the model was trained on log values. The notebook takes the log itself.

In [1]:
from pathlib import Path
import json, datetime
import numpy as np
import pandas as pd

# ---- input files ----
PRED_FILE  = "outputs/predictions.parquet"        # from the predictors notebook (csv, csv.gz or parquet)
LABEL_FILE = "outputs/lead_labels.parquet"        # from Part A (falls back to the csv.gz version)
INFO_FILE  = "outputs/labeled_buildings_info.csv" # from Part A
OUT_DIR    = Path("outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ---- column names in the predictions file (CHANGE) ----
ID_COL, TIME_COL, ACTUAL_COL = "building_id", "timestamp", "meter_reading"
# name used in this notebook : column in your predictions file
PRED_COLS = {
    "last_hour": "pred_last_hour",   # same as last hour
    "typical":   "pred_typical",     # historical typical value
    "A":         "pred_A",           # model, no history
    "B":         "pred_B",           # model, adds yesterday and last week
    "C":         "pred_C",           # model, adds previous hour
}

# ---- rules ----
USE_LOG = True                  # measure errors on log(1 + kWh)
TARGET_FALSE_ALARM = 0.005      # share of normal validation hours that may be flagged (0.5%)
MAD_FLOOR = 0.05                # smallest allowed building variation (in log units; a share of the building level if USE_LOG is False)
MIN_VAL_HOURS = 500             # buildings with fewer clean validation hours borrow values from similar buildings
FIXED_PERCENTILE = 100 * (1 - TARGET_FALSE_ALARM)   # comparison method: flag use above this percentile of the building's training use

# ---- split dates (same as the project plan) ----
TRAIN_END = pd.Timestamp("2016-09-13")
VAL_END   = pd.Timestamp("2016-11-07")

NAMES = list(PRED_COLS)
print("Predictors:", NAMES)

Predictors: ['last_hour', 'typical', 'A', 'B', 'C']


## 1. Load predictions and labels

**What happens:** the predictions are joined to the labels on building and time. Only the 200 labeled buildings are kept, because the other buildings have no labels and cannot be judged. Hours with no label inside a labeled building are treated as normal. Each hour gets a split (train, val, test) from its date.

**What to check:** the number of buildings should be about 200 and every split should have rows.

In [2]:
def read_any(path):
    """Read parquet, csv or csv.gz. If the named file is missing, try the other formats with the same name."""
    p = Path(path)
    stem = p.name.split(".")[0]
    candidates = [p, p.with_name(stem + ".csv.gz"), p.with_name(stem + ".csv"), p.with_name(stem + ".parquet")]
    for c in candidates:
        if c.exists():
            print("reading", c)
            return pd.read_parquet(c) if c.suffix == ".parquet" else pd.read_csv(c)
    raise FileNotFoundError(f"Could not find {path} (also tried csv.gz, csv, parquet). Check the folder and the name.")


pred = read_any(PRED_FILE)
need = [ID_COL, TIME_COL, ACTUAL_COL] + list(PRED_COLS.values())
missing = [c for c in need if c not in pred.columns]
if missing:
    raise ValueError(f"These columns are not in the predictions file: {missing}. Edit the settings cell. Columns found: {list(pred.columns)}")

pred = pred[need].rename(columns={ID_COL: "building_id", TIME_COL: "timestamp", ACTUAL_COL: "actual",
                                  **{v: "pred_" + k for k, v in PRED_COLS.items()}})
pred["timestamp"] = pd.to_datetime(pred["timestamp"])

labels = read_any(LABEL_FILE)[["building_id", "timestamp", "label", "in_zero_run"]]
labels["timestamp"] = pd.to_datetime(labels["timestamp"])
info = pd.read_csv(INFO_FILE)[["building_id", "site_id", "primary_use"]]

pred = pred[pred["building_id"].isin(set(info["building_id"]))]
df = (pred.merge(labels, on=["building_id", "timestamp"], how="left")
          .merge(info, on="building_id", how="left"))
df["label"] = df["label"].fillna(0).astype(int)
df["in_zero_run"] = df["in_zero_run"].fillna(False).astype(bool)
df["split"] = np.where(df["timestamp"] < TRAIN_END, "train",
                np.where(df["timestamp"] < VAL_END, "val", "test"))

print("buildings:", df["building_id"].nunique())
print("rows per split:")
print(df["split"].value_counts().reindex(["train", "val", "test"]).to_string())
print("labeled anomaly hours per split:")
print(df.groupby("split")["label"].sum().reindex(["train", "val", "test"]).to_string())

reading outputs/predictions.parquet
reading outputs/lead_labels.parquet
buildings: 200
rows per split:
split
train    1228800
val       264000
test      264000
labeled anomaly hours per split:
split
train    27274
val       4826
test      5196


## 2. Measure each predictor's error

**What happens:** for each predictor, error = log(1 + actual) minus log(1 + prediction). Positive means the building used more than expected. Negative values are clipped to zero before the log.

**Which hours count (the "valid" hours):**
- the reading is present,
- the hour is not inside a long run of zeros (treated as missing in Part A),
- **every** predictor has a prediction for that hour. This keeps all predictors on exactly the same hours, so they can be compared fairly.

Hours that are not valid are kept in the table but never flagged.

In [3]:
def tf(x):
    x = np.clip(x, 0, None)
    return np.log1p(x) if USE_LOG else x

pred_cols = ["pred_" + n for n in NAMES]
df["valid"] = df["actual"].notna() & ~df["in_zero_run"] & df[pred_cols].notna().all(axis=1)

for n in NAMES:
    df["err_" + n] = tf(df["actual"]) - tf(df["pred_" + n])

summary = df.groupby("split")["valid"].agg(valid="sum", total="size")
summary["share_valid"] = (summary["valid"] / summary["total"]).round(3)
print(summary.reindex(["train", "val", "test"]).to_string())
print()
print("Mean error per predictor on valid validation hours (should be close to 0):")
print(df[(df["split"] == "val") & df["valid"]][["err_" + n for n in NAMES]].mean().round(3).to_string())

         valid    total  share_valid
split                               
train  1135206  1228800        0.924
val     249407   264000        0.945
test    257993   264000        0.977

Mean error per predictor on valid validation hours (should be close to 0):
err_last_hour   -0.001
err_typical     -0.089
err_A           -0.024
err_B           -0.003
err_C           -0.002


## 3. Learn each building's normal error (validation only)

**What happens:** for each building and predictor, two numbers are learned from **clean validation hours** (valid, and not labeled as an anomaly):
- the **typical error** (median of the errors),
- the **normal variation** (median absolute deviation times 1.4826, which is comparable to a standard deviation but not thrown off by rare extremes).

The variation has a **floor** so a building with almost constant use does not get enormous scores from tiny wobbles.

**Small-history buildings:** a building with fewer than `MIN_VAL_HOURS` clean validation hours borrows both numbers from buildings of the same site and use type (then the same site, then all buildings).

Labeled anomaly hours are left out here only so that anomalies do not widen what counts as normal.

In [4]:
clean_val = df[(df["split"] == "val") & df["valid"] & (df["label"] == 0)]
all_ids = pd.Index(sorted(df["building_id"].unique()), name="building_id")

def mad_scaled(s):
    return (s - s.median()).abs().median() * 1.4826

stats = pd.DataFrame(index=all_ids)
stats["n_clean_val_hours"] = clean_val.groupby("building_id").size().reindex(all_ids).fillna(0).astype(int)
level = tf(clean_val["actual"]).groupby(clean_val["building_id"]).median().reindex(all_ids)
floor = MAD_FLOOR if USE_LOG else MAD_FLOOR * level.clip(lower=1e-6)

for n in NAMES:
    g = clean_val.groupby("building_id")["err_" + n]
    stats["med_" + n] = g.median().reindex(all_ids)
    stats["scale_" + n] = np.maximum(g.apply(mad_scaled).reindex(all_ids), floor)

stats = stats.join(info.set_index("building_id")[["site_id", "primary_use"]])
stats["borrowed"] = stats["n_clean_val_hours"] < MIN_VAL_HOURS

enough = stats[~stats["borrowed"]]
if len(enough) == 0:
    raise ValueError("No building has enough clean validation hours. Lower MIN_VAL_HOURS or check the split dates.")

for col in [c for c in stats.columns if c.startswith(("med_", "scale_"))]:
    by_group = enough.groupby(["site_id", "primary_use"])[col].median()
    by_site = enough.groupby("site_id")[col].median()
    key = pd.MultiIndex.from_frame(stats[["site_id", "primary_use"]])
    fill = pd.Series(by_group.reindex(key).values, index=stats.index)
    fill = fill.fillna(stats["site_id"].map(by_site)).fillna(enough[col].median())
    stats.loc[stats["borrowed"], col] = fill[stats["borrowed"]]

print("buildings with their own values:", int((~stats["borrowed"]).sum()))
print("buildings borrowing values:", int(stats["borrowed"].sum()))
print()
print("Typical normal variation (scale) across buildings, per predictor:")
print(stats[["scale_" + n for n in NAMES]].describe().loc[["min", "50%", "max"]].round(3).to_string())

buildings with their own values: 196
buildings borrowing values: 4

Typical normal variation (scale) across buildings, per predictor:
     scale_last_hour  scale_typical  scale_A  scale_B  scale_C
min            0.050          0.050    0.050    0.050    0.050
50%            0.052          0.092    0.135    0.085    0.052
max            0.277          1.129    1.192    0.786    0.349


## 4. Score every hour

**What happens:** score = |error minus the building's typical error| divided by the building's normal variation. A score of 3 means "this hour's error is about three normal-sized wobbles away from what is usual for this building." The same scoring function is used again in step 8 on the frozen values.

In [5]:
def add_scores(frame, stats_table):
    """Add score_<name> columns using a table of building values. Invalid hours get NaN."""
    out = frame.merge(stats_table[[c for c in stats_table.columns if c.startswith(("med_", "scale_"))]],
                      left_on="building_id", right_index=True, how="left")
    for n in NAMES:
        s = (out["err_" + n] - out["med_" + n]).abs() / out["scale_" + n]
        out["score_" + n] = s.where(out["valid"])
    return out.drop(columns=[c for c in out.columns if c.startswith(("med_", "scale_"))])

scored_val = add_scores(df[df["split"] == "val"], stats)
cv = scored_val[scored_val["valid"] & (scored_val["label"] == 0)]
print("Score summary on clean validation hours:")
print(cv[["score_" + n for n in NAMES]].describe(percentiles=[.5, .9, .99, .995]).round(2).to_string())

Score summary on clean validation hours:
       score_last_hour  score_typical    score_A    score_B    score_C
count        244690.00      244690.00  244690.00  244690.00  244690.00
mean              1.13           1.78       1.49       1.74       0.95
std               3.08           7.99       5.74       6.71       2.35
min               0.00           0.00       0.00       0.00       0.00
50%               0.52           0.64       0.67       0.66       0.55
90%               2.49           2.05       1.98       2.41       1.90
99%               9.75          36.52      21.72      27.16       6.42
99.5%            13.46          62.44      45.39      52.88       9.64
max             176.84         128.13     124.98     125.24     126.17


## 5. Pick one cut-off per predictor (same false-alarm rate)

**What happens:** for each predictor, the cut-off is the score that only `TARGET_FALSE_ALARM` (0.5%) of clean validation hours exceed. Every predictor therefore raises about the same number of false alarms on normal data, and the comparison later is about **how many real events each one catches at that equal cost**.

Labeled anomaly hours are left out of this step only.

In [6]:
cutoffs = {n: float(np.quantile(cv["score_" + n].dropna(), 1 - TARGET_FALSE_ALARM)) for n in NAMES}
print("Cut-offs (score above this is flagged):")
for n, c in cutoffs.items():
    print(f"  {n:10s} {c:6.2f}")

Cut-offs (score above this is flagged):
  last_hour   13.46
  typical     62.44
  A           45.39
  B           52.88
  C            9.64


## 6. The comparison method: fixed percentile of the building's own use

**What happens:** for each building, take its **training** hours (valid, not labeled as an anomaly) and find the value that `FIXED_PERCENTILE` (99.5%) of them stay below. Later, any hour above that value is flagged. It uses no model, so it shows what a simple rule achieves.

It only flags unusually **high** use, because it looks at the level and not at an error. The percentile is set from the target rate so the comparison starts from a similar false-alarm budget, but its real rate on validation will differ and is shown in step 9.

If the predictions file has no training rows, this step is skipped with a message.

In [7]:
train_clean = df[(df["split"] == "train") & df["actual"].notna() & ~df["in_zero_run"] & (df["label"] == 0)]
if len(train_clean) == 0:
    print("No training rows in the predictions file. The fixed-percentile method is skipped.")
    print("To include it, save the training rows (prediction columns may be empty) in the predictions file, or load the hourly table here.")
    fixed = None
else:
    fixed = (train_clean.groupby("building_id")["actual"].quantile(FIXED_PERCENTILE / 100)
             .reindex(all_ids).rename("fixed_threshold").to_frame())
    print("buildings with a fixed threshold:", int(fixed["fixed_threshold"].notna().sum()), "of", len(fixed))
    print(fixed["fixed_threshold"].describe().round(1).to_string())

buildings with a fixed threshold: 200 of 200
count     200.0
mean      283.7
std       523.9
min         3.2
25%        65.8
50%       161.0
75%       315.2
max      6389.5


## 7. Freeze everything

**What happens:** the building values, the cut-offs and the fixed thresholds are written to files, together with the settings and the time they were saved. **From here on, nothing is recomputed.** If you change a setting, rerun the notebook from the top and treat it as a new version, and do it before looking at any test result.

In [8]:
stats.to_csv(OUT_DIR / "cut_building_values.csv")
if fixed is not None:
    fixed.to_csv(OUT_DIR / "cut_fixed_thresholds.csv")
frozen = {
    "created": datetime.datetime.now().isoformat(timespec="seconds"),
    "cutoffs": cutoffs,
    "settings": {"USE_LOG": USE_LOG, "TARGET_FALSE_ALARM": TARGET_FALSE_ALARM, "MAD_FLOOR": MAD_FLOOR,
                 "MIN_VAL_HOURS": MIN_VAL_HOURS, "FIXED_PERCENTILE": FIXED_PERCENTILE,
                 "TRAIN_END": str(TRAIN_END), "VAL_END": str(VAL_END), "predictors": NAMES},
}
with open(OUT_DIR / "cutoffs.json", "w") as f:
    json.dump(frozen, f, indent=2)
print("frozen at", frozen["created"])
print("saved: cut_building_values.csv, cutoffs.json" + (", cut_fixed_thresholds.csv" if fixed is not None else ""))

frozen at 2026-10-19T00:47:38
saved: cut_building_values.csv, cutoffs.json, cut_fixed_thresholds.csv


## 8. Produce the flags from the frozen files

**What happens:** the saved files are loaded again (not the variables from above) and used to score and flag every validation and test hour. An hour is flagged when its score is above its predictor's cut-off. The fixed method flags an hour when the reading is above the building's fixed threshold. Hours that are not valid are never flagged.

The output table has one row per building and hour, with the label, the score and the flag for each method. This is the input to the evaluation notebook.

In [9]:
stats_f = pd.read_csv(OUT_DIR / "cut_building_values.csv", index_col="building_id")
with open(OUT_DIR / "cutoffs.json") as f:
    frozen_f = json.load(f)
fixed_path = OUT_DIR / "cut_fixed_thresholds.csv"
fixed_f = pd.read_csv(fixed_path, index_col="building_id") if fixed_path.exists() else None

out = add_scores(df[df["split"].isin(["val", "test"])].copy(), stats_f)
for n in NAMES:
    out["flag_" + n] = out["valid"] & (out["score_" + n] > frozen_f["cutoffs"][n])
if fixed_f is not None:
    thr = out["building_id"].map(fixed_f["fixed_threshold"])
    out["flag_fixed"] = out["valid"] & (out["actual"] > thr)

keep = (["building_id", "timestamp", "split", "label", "valid", "actual"]
        + ["score_" + n for n in NAMES] + ["flag_" + n for n in NAMES]
        + (["flag_fixed"] if fixed_f is not None else []))
flags = out[keep].sort_values(["building_id", "timestamp"]).reset_index(drop=True)

try:
    flags.to_parquet(OUT_DIR / "flags.parquet", index=False)
    print("saved flags.parquet")
except Exception as e:
    print("parquet not available (", type(e).__name__, "), saving csv.gz instead")
    flags.to_csv(OUT_DIR / "flags.csv.gz", index=False)

flag_cols = [c for c in flags.columns if c.startswith("flag_")]
print()
print("Flagged hours per split (counts only, not a result):")
print(flags.groupby("split")[flag_cols].sum().to_string())

saved flags.parquet

Flagged hours per split (counts only, not a result):
       flag_last_hour  flag_typical  flag_A  flag_B  flag_C  flag_fixed
split                                                                  
test             1098           291     353     583    1856        4944
val              1506          1315    1268    1313    1724        2656


## 9. Checks before moving on

**Check 1, flag rate on normal validation hours.** For the model-based predictors this should be very close to the target (0.5%). If one is far off, there is a bug in steps 3 to 5. The fixed method will not match exactly, because it uses a level rule and not an error rule.

**Check 2, order of files.** The cut-off file must be older than the flags file.

**Remember:** these rates are set by the cut-off. They say nothing about which predictor is better.

In [10]:
chk = flags[(flags["split"] == "val") & flags["valid"] & (flags["label"] == 0)]
rates = (chk[flag_cols].mean() * 100).round(3).rename("false_alarm_%")
print(f"Target: {TARGET_FALSE_ALARM*100:.3f}%")
print(rates.to_string())

bad = [c for c in flag_cols if c != "flag_fixed" and abs(chk[c].mean() - TARGET_FALSE_ALARM) > TARGET_FALSE_ALARM * 0.2]
print()
print("Predictors more than 20% away from the target:", bad if bad else "none")

flag_file = OUT_DIR / ("flags.parquet" if (OUT_DIR / "flags.parquet").exists() else "flags.csv.gz")
t_cut = (OUT_DIR / "cutoffs.json").stat().st_mtime
t_flag = flag_file.stat().st_mtime
print("cut-offs saved before flags:", t_cut <= t_flag)

Target: 0.500%
flag_last_hour    0.495
flag_typical      0.499
flag_A            0.500
flag_B            0.500
flag_C            0.500
flag_fixed        1.057

Predictors more than 20% away from the target: none
cut-offs saved before flags: True


## What comes next

The next notebook (Part D, evaluation) takes `flags` together with `lead_events.csv` and measures, per predictor and per event length group:
- event recall (an event counts as found if at least one of its hours is flagged),
- event precision (flagged runs that touch a real event),
- detection delay,
- precision at the top 50 and 100,
- all labeled buildings and Education only.

Files created by this notebook (all in `outputs`): `cut_building_values.csv`, `cutoffs.json`, `cut_fixed_thresholds.csv`, `flags.parquet` (or `flags.csv.gz`).